# Endgame NLP: Text Classification

This notebook demonstrates Endgame's NLP capabilities:
1. Load the 20 Newsgroups text classification dataset
2. Clean and preprocess text with `TextPreprocessor`
3. Train a `TransformerClassifier` (HuggingFace-backed)
4. Evaluate with NLP metrics
5. Use the high-level `TextPredictor` API

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## 1. Load 20 Newsgroups

A classic multi-class text classification benchmark with ~18,000 newsgroup
posts across 20 categories. We use a 4-category subset for speed.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

categories = ["sci.space", "comp.graphics", "rec.sport.baseball", "talk.politics.misc"]

train_raw = fetch_20newsgroups(
    subset="train", categories=categories, remove=("headers", "footers", "quotes"),
)
test_raw = fetch_20newsgroups(
    subset="test", categories=categories, remove=("headers", "footers", "quotes"),
)

df_train = pd.DataFrame({"text": train_raw.data, "label": train_raw.target})
df_test = pd.DataFrame({"text": test_raw.data, "label": test_raw.target})

label_names = train_raw.target_names

print(f"Train: {len(df_train)}, Test: {len(df_test)}")
print(f"Classes: {label_names}")
print(f"\nLabel distribution (train):")
print(df_train["label"].value_counts().sort_index())
print(f"\nSample text (first 200 chars):")
print(df_train["text"].iloc[0][:200])

## 2. Text Preprocessing

`TextPreprocessor` provides cleaning utilities for common text issues:
HTML tags, URLs, emails, unicode normalization, and whitespace cleanup.

In [ ]:
from endgame.nlp import TextPreprocessor
from endgame.nlp.preprocessing import clean_html, clean_urls, clean_email, normalize_whitespace

# Demo standalone cleaning functions
dirty_text = "  Check <b>this</b> out: https://example.com  \n\n  contact me@test.org  "
print(f"Original:    '{dirty_text}'")
print(f"Clean HTML:  '{clean_html(dirty_text)}'")
print(f"Clean URLs:  '{clean_urls(dirty_text)}'")
print(f"Clean email: '{clean_email(dirty_text)}'")
print(f"Whitespace:  '{normalize_whitespace(dirty_text)}'")

# TextPreprocessor uses a builder pattern to chain steps:
preprocessor = (
    TextPreprocessor()
    .clean_html()
    .clean_urls()
    .clean_email()
    .normalize_whitespace()
)
print(f"\nAll steps combined: '{preprocessor(dirty_text)}'")

In [ ]:
# Apply batch preprocessing to the training data
texts_clean = preprocessor.batch(df_train["text"].tolist())
df_train["text_clean"] = texts_clean

# Show before/after for a sample
idx = 5
print(f"Before ({len(df_train['text'].iloc[idx])} chars):")
print(df_train["text"].iloc[idx][:150])
print(f"\nAfter ({len(df_train['text_clean'].iloc[idx])} chars):")
print(df_train["text_clean"].iloc[idx][:150])

In [ ]:
# Also available: kaggle_preset() for competition text cleaning
kaggle_pp = TextPreprocessor.kaggle_preset()
print(f"Kaggle preset type: {type(kaggle_pp).__name__}")

## 3. TransformerClassifier

`TransformerClassifier` wraps HuggingFace transformers with a sklearn-compatible
`fit`/`predict`/`predict_proba` interface. It handles tokenization, training
loops, learning rate scheduling, and mixed-precision training.

In [ ]:
from endgame.nlp import TransformerClassifier

# Show the available default models
from endgame.automl.text import DEFAULT_MODELS
print("Default transformer models:")
for key, model_name in DEFAULT_MODELS.items():
    print(f"  {key:>15s}: {model_name}")

In [ ]:
# Train DistilBERT on the 4-class newsgroups task
# Using a small subset and few epochs for demo speed
clf = TransformerClassifier(
    model_name="distilbert-base-uncased",
    num_labels=len(label_names),
    max_length=256,
    num_epochs=2,
    batch_size=16,
    learning_rate=3e-5,
    verbose=True,
)

# Use cleaned text, take a 500-sample subset for speed
train_subset = df_train.sample(500, random_state=42)
X_train_texts = train_subset["text_clean"].tolist()
y_train_labels = train_subset["label"].values

print(f"Training on {len(X_train_texts)} samples...")
clf.fit(X_train_texts, y_train_labels)

In [ ]:
# Evaluate on test set
texts_test_clean = preprocessor.batch(df_test["text"].tolist())
y_pred = clf.predict(texts_test_clean)
y_proba = clf.predict_proba(texts_test_clean)

acc = accuracy_score(df_test["label"].values, y_pred)
print(f"Test accuracy: {acc:.4f}")
print(f"Probability shape: {y_proba.shape}")
print(f"\n{classification_report(df_test['label'].values, y_pred, target_names=label_names)}")

## 4. TextPredictor (High-Level AutoML API)

`TextPredictor` automates the entire NLP pipeline: text column detection,
preprocessing, model selection (DistilBERT, DeBERTa, etc.), training,
and optional domain-adaptive pretraining (DAPT) and pseudo-labeling.

In [ ]:
from endgame.automl import TextPredictor

# The TextPredictor expects a DataFrame with text and label columns
print("TextPredictor capabilities:")
print("  Models: distilbert (fast), deberta-v3-base (medium), deberta-v3-large (best)")
print("  Features: DAPT, pseudo-labeling, auto text column detection")
print("  Interface: fit(df) -> predict(df) -> predict_proba(df)")

# Example usage:
# predictor = TextPredictor(
#     label="label",
#     text_column="text",
#     presets="fast",
#     time_limit=300,
# )
# predictor.fit(df_train)
# preds = predictor.predict(df_test)

## 5. Pseudo-Labeling

Semi-supervised learning via pseudo-labeling: train on labeled data,
predict confident pseudo-labels on unlabeled data, retrain on both.

In [ ]:
from endgame.nlp import PseudoLabelTrainer

# PseudoLabelTrainer wraps any classifier with iterative pseudo-labeling
print("PseudoLabelTrainer parameters:")
print("  confidence_threshold: 0.95 (only pseudo-label high-confidence predictions)")
print("  n_iterations: 3 (number of pseudo-labeling rounds)")
print("  balance_classes: True (maintain class balance in pseudo-labels)")
print()
print("Usage:")
print("  trainer = PseudoLabelTrainer(confidence_threshold=0.95, n_iterations=3)")
print("  trainer.fit(base_estimator, X_labeled, y_labeled, X_unlabeled)")

## Summary

In this notebook we:
- Loaded the 20 Newsgroups dataset (real text data, 4 categories)
- Cleaned text with `TextPreprocessor` (HTML, URLs, emails, whitespace)
- Trained a DistilBERT classifier using `TransformerClassifier`
- Evaluated with accuracy and per-class metrics
- Reviewed the high-level `TextPredictor` API for automated NLP
- Explored pseudo-labeling for semi-supervised learning

Next steps:
- Try `09_multimodal.ipynb` for combining text, image, and tabular data
- Use `use_dapt=True` with `TextPredictor` for domain-adaptive pretraining